# Baseline Model Evaluation
This notebook evaluates the performance of the base instruction-tuned language model before fine-tuning.

The objective is to establish a baseline performance level and later compare it with the fine-tuned model.

In this notebook we will:

- Load the instruction-tuned transformer model.
- Load the held-out test dataset.
- Format prompts for evaluation.
- Generate predicted scores and rationales.
- Parse model outputs.
- Compare predictions with ground-truth labels.
- Save results for later analysis.

This baseline evaluation is essential to measure the actual impact of fine-tuning.

## 1. Import Libraries and Configure Environment
we import all required libraries for inference and evaluation.

In [1]:
import json
import torch
import re  
import pandas as pd
from pathlib import Path
from transformers import (AutoTokenizer , AutoModelForCausalLM)
from sklearn.metrics import accuracy_score, mean_absolute_error, cohen_kappa_score

print("PyTorch Version:", torch.__version__)

device = ("cuda"if torch.cuda.is_available()else "cpu")
print("Device:", device)

if device == "cuda":
    print("GPU:",torch.cuda.get_device_name(0))

import sys
sys.path.append("..")

from src.prompts import build_prompt

PyTorch Version: 2.5.1+cu121
Device: cuda
GPU: NVIDIA A100-SXM4-80GB


## 2. Load Baseline Model
In this section, we load the pretrained instruction-tuned transformer model.

This model serves as the baseline system and will be evaluated before any additional training.

The purpose of this step is to establish a reference performance level that will later be compared with the fine-tuned version.

The model will generate:

- A score (0–4)
- A rationale explaining the evaluation

In [2]:
MODEL_NAME = ("unsloth/mistral-7b-instruct-v0.2-bnb-4bit")

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto"
)

print("Model loaded successfully.")

Loading tokenizer...


Loading model...


Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.


model.safetensors:   0%|          | 0.00/4.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

Model loaded successfully.


## 3. Load Test Dataset
In this section, we load the held-out test dataset.

The test set is kept completely separate from training and validation data because its purpose is to provide an unbiased estimate of model performance.

Only the test split will be used in this notebook.

Each sample contains:

- task
- reference
- submission
- rubric
- ground-truth score
- rationale

These labels will later be compared against the model predictions.

In [3]:
test_path = Path("../data/test.jsonl")

records = []

with open(test_path,"r",encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

test_df = pd.DataFrame(records)

print(f"Loaded {len(test_df)} test samples.")
print("Sample test data:")
display(test_df.head(2))

Loaded 20 test samples.
Sample test data:


,task,reference,submission,rubric,score,rationale,reference_length,submission_length,rationale_length
0,Customer received a duplicate order and asks w...,We apologize that you received a duplicate ord...,We apologize that you received a duplicate ord...,"{'1': 'Shows understanding and apology', '2': ...",4,The reply is complete and professional because...,45,43,16
1,Customer complains about poor product quality.,We are sorry to hear that the product did not ...,The product may not be defective. Sometimes is...,"{'1': 'Shows understanding and apology', '2': ...",2,The response provides a next step but lacks em...,43,22,17


## 4. Build Evaluation Prompt
create the prompt template used to evaluate customer support replies.

Prompt design is an important part of instruction-tuned language models because it determines how information is presented to the model and how outputs are generated.

The prompt includes:

- Task description
- Reference response
- Candidate submission
- Evaluation rubric

The model is instructed to act as an expert evaluator and generate:

- A score from 0 to 4
- A short rationale

To make outputs easier to process later, the model will be instructed to return valid JSON only.

In [4]:
# src/prompts.py
sample_prompt = build_prompt(test_df.iloc[0])
print(sample_prompt)

You are an expert evaluator for customer support replies.
Evaluate the quality of the submission.
Return ONLY valid JSON.

Format:
{
    "score": integer from 0 to 4, where 0 is the worst and 4 is the best,
    "rationale": "short explanation"
}

Task:
Customer received a duplicate order and asks what to do.

Reference:
We apologize that you received a duplicate order. We will review the order and payment details to confirm whether this was an error. If you were charged twice, we will arrange a refund, and we will provide instructions for returning the duplicate item if needed.

Submission:
We apologize that you received a duplicate order. We will review the order and payment details to confirm whether this was an error. If you were charged twice, we will arrange a refund and provide instructions for returning the duplicate item if needed.

Rubric:
1. Shows understanding and apology
2. Provides correct and relevant information
3. Provides clear solution or next step
4. Uses polite and 

## 5. Run Baseline Inference
In this section, we run the pretrained model on the held-out test set.

For each sample:

- Build the evaluation prompt
- Generate a response
- Store the prediction

At this stage, the model has not been fine-tuned on our dataset.

The generated outputs will later be compared with the human-annotated scores.

In [5]:
def generate_response(prompt):
    inputs = tokenizer(prompt ,return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=150, do_sample=False)

    response = tokenizer.decode(outputs[0],skip_special_tokens=True)

    return response

example = build_prompt(test_df.iloc[1])
result = generate_response(example)

print(result)

You are an expert evaluator for customer support replies.
Evaluate the quality of the submission.
Return ONLY valid JSON.

Format:
{
    "score": integer from 0 to 4, where 0 is the worst and 4 is the best,
    "rationale": "short explanation"
}

Task:
Customer complains about poor product quality.

Reference:
We are sorry to hear that the product did not meet your expectations. We take quality issues seriously and would like to help resolve this. Please share more details or images so we can investigate and offer a replacement or refund if eligible.

Submission:
The product may not be defective. Sometimes issues happen due to usage. Please send a photo so we can check the situation.

Rubric:
1. Shows understanding and apology
2. Provides correct and relevant information
3. Provides clear solution or next step
4. Uses polite and professional tone

Evaluation:
{
    "score": 2,
    "rationale": "The response does not show a clear understanding of the issue and does not offer a satisfact

## 6. Parse Model Output
The model output may contain extra text before the final prediction.

In this section, we extract only the JSON object containing:

- score
- rationale

This ensures that predictions can be processed automatically and compared against the ground-truth labels.

In [6]:
def extract_prediction(text):
    try:
        matches = re.findall(r"\{[\s\S]*?\}", text)
        if matches:
            last_json = matches[-1]
            return json.loads(last_json)

    except Exception:
        pass

    return {
        "score": None,
        "rationale": None
    }

parsed = extract_prediction(result)
print(parsed)

{'score': 2, 'rationale': "The response does not show a clear understanding of the issue and does not offer a satisfactory solution. It suggests that the issue might be due to usage and asks for a photo, which may not be helpful in determining the cause of the problem. A more empathetic and solution-oriented response would be more effective in addressing the customer's concern."}


## 7. Run Evaluation on Test Set
In this step, we evaluate the model on the held-out test set.

For each sample in the test dataset:

We build a structured prompt
We generate a response using the model
We extract the predicted score and rationale
We store the results for later analysis

This allows us to compare the model’s predictions against the ground-truth labels and measure performance objectively.

In [7]:
predictions = []

for _, row in test_df.iterrows():
    prompt = build_prompt(row)
    result = generate_response(prompt)
    parsed = extract_prediction(result)

    predictions.append({
        "task": row["task"],
        "reference": row["reference"],
        "submission": row["submission"],
        "true_score": row["score"],
        "pred_score": parsed["score"],
        "rationale": parsed["rationale"]
    })

pred_df = pd.DataFrame(predictions)

print("Evaluation completed.")
display(pred_df[["true_score", "pred_score"]].head(10))

Evaluation completed.


,true_score,pred_score
0,4,4
1,2,2
2,1,0
3,1,1
4,3,3
5,2,2
6,4,4
7,3,3
8,0,0
9,0,0


## 8. Evaluation Metrics
Now we compute evaluation metrics to measure how close the model predictions are to the ground truth.

We use:

- Accuracy (exact match between predicted and true score)
- Mean Absolute Error (MAE)

These metrics help us understand both correctness and distance from the expected score.

In [8]:
accuracy = accuracy_score(pred_df["true_score"], pred_df["pred_score"])
mae = mean_absolute_error(pred_df["true_score"], pred_df["pred_score"])

qwk = cohen_kappa_score(
    pred_df["true_score"],
    pred_df["pred_score"],
    weights="quadratic"
)

print(f"Accuracy: {accuracy*100:.2f}%")
print(f"MAE: {mae:.2f}")
print(f"Quadratic Weighted Kappa: {qwk:.2f}")

Accuracy: 80.00%
MAE: 0.20
Quadratic Weighted Kappa: 0.95


## 9. Error Analysis (Incorrect Predictions Review)
In this section, we analyze the model’s mistakes to better understand its behavior.

Instead of only looking at evaluation metrics, we inspect individual samples where the model prediction differs from the ground truth.

This helps us identify:

- Common failure patterns
- Whether the model is biased toward higher or lower scores
- Cases where the rubric interpretation is unclear
- Types of responses that are difficult for the model to evaluate

Error analysis is important because it provides qualitative insight beyond numerical metrics.

In [9]:
errors_df = pred_df[pred_df["true_score"] != pred_df["pred_score"]]

print(f"Number of errors: {len(errors_df)}")
display(errors_df[["true_score", "pred_score"]].head(10))

Number of errors: 4


,true_score,pred_score
2,1,0
12,1,2
13,3,4
19,3,4


## 10. Visualizing Some Error Examples
Here we inspect a few misclassified examples to understand why the model made incorrect predictions.

We compare:

- True score
- Predicted score
- Model rationale

This helps us evaluate whether the mistakes are due to misunderstanding of the rubric or natural ambiguity in the data.

In [10]:
for i, row in errors_df.head(1).iterrows():
    print("=" * 80)
    print("TASK:")
    print(row["task"])
    print("\nREFERENCE:")
    print(row["reference"])
    print("\nSUBMISSION:")
    print(row["submission"])
    print("\nTRUE SCORE:", row["true_score"])
    print("PREDICTED SCORE:", row["pred_score"])
    print("\nRATIONALE:")
    print(row["rationale"])

TASK:
Customer wants to delay the delivery date because they will not be home.

REFERENCE:
We understand that you would like to delay the delivery date. We will check whether the shipment can still be rescheduled with the courier. If rescheduling is available, we will help update the delivery date. If not, we will suggest the best available delivery options.

SUBMISSION:
You should plan to be home when the order arrives.

TRUE SCORE: 1
PREDICTED SCORE: 0

RATIONALE:
The submission does not acknowledge the customer's request and instead gives a general advice. It does not provide any helpful information or solution.


## 11. Save Results

In this step, we save the model predictions and evaluation results to disk.
This allows us to reuse the outputs in further analysis and fine-tuning comparison without recomputing inference.

In [11]:
output_path = Path("../data/baseline_predictions.jsonl")
pred_df.to_json(output_path, orient="records", lines=True)
print(f"Results saved to {output_path}")

Results saved to ../data/baseline_predictions.jsonl


## 12. Summary

In this notebook, we built a complete baseline evaluation system for a customer support reply scoring task using a pre-trained instruction-tuned language model.

The main goal was to measure how well a general-purpose LLM can evaluate customer support responses before any fine-tuning, and to establish a reference point that can later be compared with the fine-tuned model.

### 1. Environment Setup and Model Loading

We started by importing the required libraries, including PyTorch, Pandas, Transformers, and Scikit-learn. We also checked GPU availability to make sure inference runs efficiently.

The baseline model used in this notebook is:

- Mistral 7B Instruct
- 4-bit quantized version for memory efficiency

The model and tokenizer were loaded successfully on the GPU, which allowed us to run inference on the test set efficiently.

### 2. Dataset Loading

After expanding the full dataset to 200 samples, we evaluated the baseline model on the held-out test split.

The test set contains 20 samples and remains balanced across all score classes:

- Score 0: 4 samples
- Score 1: 4 samples
- Score 2: 4 samples
- Score 3: 4 samples
- Score 4: 4 samples

Each test sample includes the required fields:

- task
- reference
- submission
- rubric
- score
- rationale

Using a separate test set is important because it allows us to evaluate the model on unseen examples and measure its performance fairly.

### 3. Prompt Construction

We designed a structured evaluation prompt that includes the task description, reference answer, candidate submission, and rubric.

The model was instructed to act as an expert evaluator and return only a valid JSON object containing:

- A predicted score from 0 to 4
- A short rationale explaining the prediction

This prompt structure is important because the quality of the prompt directly affects how well the language model follows the evaluation task.

### 4. Baseline Inference

For each test sample, we generated an evaluation prompt and passed it to the baseline model. The model generated a predicted score and rationale without any fine-tuning on our dataset.

We used deterministic generation by setting sampling to false, which makes the predictions more stable and easier to compare.

### 5. Output Parsing

Because language models may sometimes generate extra text, we implemented a parsing step to extract the JSON object from the model output.

This allowed us to automatically retrieve the predicted score and rationale for each test sample and store them in a structured format.

### 6. Evaluation Metrics

We compared the predicted scores with the ground-truth human scores using three metrics:

- Accuracy: measures exact score matches
- MAE: measures the average distance between predicted and true scores
- Quadratic Weighted Kappa: measures agreement while considering how far wrong predictions are

The baseline model achieved:

- Accuracy: 80.00%
- MAE: 0.20
- Quadratic Weighted Kappa: 0.95

These results show that the baseline model performs well overall and usually predicts scores close to the correct labels.

### 7. Error Analysis

We also analyzed the incorrect predictions instead of only depending on numerical metrics.

The model made 4 incorrect predictions out of 20 test samples. Most of the mistakes were small differences, such as predicting a score one level higher or lower than the true score.

This error analysis helps us understand where the baseline model is still inconsistent and why fine-tuning may improve alignment with our specific rubric.

### 8. Results Storage

Finally, we saved the baseline predictions into a JSONL file:

- data/baseline_predictions.jsonl

This file contains the original sample information, the true score, the predicted score, and the model-generated rationale. Saving these outputs makes the results reusable for later comparison with the fine-tuned model.

### Final Conclusion

This notebook establishes a complete baseline evaluation pipeline for a rubric-based customer support reply evaluator.

After expanding the dataset to 200 samples, the baseline model was evaluated on a balanced held-out test set of 20 samples. The base instruction-tuned model achieved 80.00% accuracy, 0.20 MAE, and 0.95 quadratic weighted kappa.

These results show that the base model already understands the general evaluation task reasonably well. However, it still made 4 incorrect predictions, mostly with small score differences. This confirms that fine-tuning is still useful to better align the model with our specific rubric, dataset style, and scoring expectations.

Overall, this baseline provides a strong reference point for evaluating whether the fine-tuned model improves performance in the next stage.